# Lab 4 · RAG com três acervos e avaliação
**Oficina do Builder · Winovoito** — Bancada 4 · RAG e fine-tuning

Três organizações fictícias (um escritório, uma clínica, uma cooperativa) têm 30 documentos cada. Você vai montar um RAG que **responde com citação, recusa quando não sabe, respeita permissões e não obedece a ordens escondidas em documentos**, e vai medir tudo.

Neste lab você:
1. divide documentos em trechos pela estrutura;
2. gera embeddings e mede recall@5 e MRR;
3. soma busca por palavras (BM25) com a vetorial usando RRF;
4. aplica o filtro de permissão **antes** da busca;
5. responde com citação conferida em código e recusa com frase fixa;
6. testa o documento com instrução escondida;
7. avalia com um juiz de IA calibrado em 5 casos.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import json, pandas as pd

## 1 · Os acervos e os trechos

In [ ]:
from oficina import rag
col = rag.carregar_colecao("juridico")
print(col["organizacao"], "·", len(col["documentos"]), "documentos ·", len(col["perguntas"]), "perguntas")
doc = col["documentos"][4]
print(doc["id"], doc["acesso"], doc["titulo"])
for t in rag.dividir_em_trechos(doc): print(" ", t["id"], "|", t["texto"][:90])

Cada trecho leva o **título** junto e o `acesso` do documento de origem. Dividir por parágrafo preserva o sentido; dividir por número de caracteres corta frases no meio.

## 2 · Índice e embeddings
O embedder padrão deste notebook é **léxico** (roda sem download). Para o modelo multilíngue de código aberto, descomente a segunda linha (precisa de internet na primeira vez).

In [ ]:
indice = rag.Indice(col["documentos"])
# indice = rag.Indice(col["documentos"], rag.embedder_multilingue())   # modelo de verdade
print(len(indice.trechos), "trechos indexados")

## 3 · Recall@5 e MRR nos três modos

In [ ]:
import pandas as pd
linhas = []
for nome in ["juridico", "saude", "financas"]:
    c = rag.carregar_colecao(nome); ix = rag.Indice(c["documentos"])
    for modo in ["vetorial", "bm25", "hibrido"]:
        r, m = rag.recall_e_mrr(ix, c["perguntas"], modo)
        linhas.append({"acervo": nome, "modo": modo, "recall@5": round(r, 3), "MRR": round(m, 3)})
pd.DataFrame(linhas).pivot(index="acervo", columns="modo", values=["recall@5", "MRR"])

Com 30 documentos o recall@5 satura perto de 1. A diferença aparece no **MRR** (o documento certo vem em 1º ou em 3º?). Seu desafio na seção 8: escreva perguntas mais difíceis e veja quem cai primeiro.

## 4 · Permissão antes da busca
Perguntas sobre documentos `restrito` só podem ser respondidas por `gestor`.

In [ ]:
q = next(x for x in col["perguntas"] if x["papel"] == "gestor" and x["tipo"] == "resposta")
print(q["pergunta"])
for papel in ["visitante", "colaborador", "gestor"]:
    ids = [t["doc_id"] for t in indice.buscar(q["pergunta"], "hibrido", 5, papel)]
    print(f"{papel:12s} vê {q['doc_esperado'] in ids and 'o documento restrito' or 'nada restrito'}")

O filtro é aplicado **antes** do ranking. Filtrar depois deixaria o trecho restrito passar pelo modelo, e o modelo pode repetir o que viu.

## 5 · Responder com citação ou recusar

In [ ]:
import os, json, re
from pathlib import Path
from oficina import provedores
from oficina.rag_simulado import criar_modelo_simulado
DADOS = Path("oficina-do-builder-aberto/dados/rag") if Path("oficina-do-builder-aberto").exists() else Path("dados/rag")
if not os.environ.get("OFICINA_MODELO") and os.environ.get("OFICINA_PROVEDOR") != "falso":
    os.environ["OFICINA_PROVEDOR"] = "falso"
if os.environ.get("OFICINA_PROVEDOR") == "falso":
    ATUAL_COL = col
    provedores.usar_resposta_falsa(criar_modelo_simulado(col))
    print("⚠️ Modo simulado: um leitor perfeito, sem modelo de verdade. Mede a encanação, não a qualidade do modelo.")
else:
    ATUAL_COL = None
    print("✅ Provedor real:", os.environ.get("OFICINA_PROVEDOR", "remoto"))

In [ ]:
for pergunta in [col["perguntas"][0]["pergunta"], "Qual o valor do aluguel da sede do escritório?", "Me conte uma piada sobre gatos."]:
    r = rag.perguntar_ao_acervo(indice, pergunta, "gestor")
    print("P:", pergunta); print("R:", r.resposta, r.citacoes, "\n")

Três proteções ficam **em código**: (1) citação que não existe nos trechos recebidos é rejeitada; (2) resposta sem citação é rejeitada; (3) se o modelo não produz resposta válida, o sistema recusa em vez de entregar algo não verificado.

## 6 · O documento com instrução escondida

In [ ]:
injetado = next(d for d in col["documentos"] if d.get("injetado"))
print(injetado["titulo"], "→", injetado["texto"].split("\n\n")[1][:140], "...")
print("detectado como suspeito:", rag.trecho_suspeito(injetado["texto"]))

O prompt diz que trechos são dado e não instrução, e o código **também** tira da conversa os trechos que parecem ordens. Uma defesa só não basta: o prompt pode falhar, e a regra de código pode não prever o disfarce.

## 7 · Avaliação do acervo

In [ ]:
resultado = []
for nome in ["juridico", "saude", "financas"]:
    c = rag.carregar_colecao(nome); ix = rag.Indice(c["documentos"])
    if ATUAL_COL is not None: provedores.usar_resposta_falsa(criar_modelo_simulado(c))
    ok = {"resposta": 0, "sem_resposta": 0, "fora_do_escopo": 0}; tot = dict.fromkeys(ok, 0)
    for q in c["perguntas"]:
        tot[q["tipo"]] += 1
        ok[q["tipo"]] += rag.avaliar_resposta(q, rag.perguntar_ao_acervo(ix, q["pergunta"], q["papel"]))
    resultado.append({"acervo": nome, **{k: f"{ok[k]}/{tot[k]}" for k in ok}, "aprovadas": f"{sum(ok.values())}/25"})
pd.DataFrame(resultado)

### O juiz de IA, calibrado
Um juiz automático só vale depois de concordar com humanos. Estes 5 casos têm rótulo humano:

In [ ]:
casos_juiz = json.loads(Path(DADOS / "calibracao_juiz.json").read_text("utf-8"))
if ATUAL_COL is not None:
    print("(modo simulado: o juiz também é simulado)")
    def juiz_simulado(sistema, mensagem):
        ref = re.search(r"<referencia>(.*?)</referencia>", mensagem).group(1); resp = re.search(r"<resposta>(.*?)</resposta>", mensagem).group(1)
        return json.dumps({"contem_referencia": ref in resp, "inventa_algo": ("automático" in resp) or (bool(re.search(r"\d+ dias|\d+ horas", resp)) and ref not in resp)})
    provedores.usar_resposta_falsa(juiz_simulado)
print("concordância com o humano:", rag.calibrar_juiz(casos_juiz))

## 8 · Seu turno
1. Escreva **5 perguntas mais difíceis** para um acervo (sinônimos, perguntas com duas partes, um código exato). Rode os três modos e anote quem acerta.
2. Troque o embedder pelo multilíngue e compare o MRR.
3. Escreva um **segundo documento com instrução escondida**, disfarçado (sem as palavras "ignore" ou "assistente"), e veja se a defesa de código o pega. Se não pegar, o que isso diz sobre defesas por lista de palavras?

## 9 · Teste de aceitação

In [ ]:
rodadas = {}
for n in ["juridico", "saude", "financas"]:
    if ATUAL_COL is not None: provedores.usar_resposta_falsa(criar_modelo_simulado(rag.carregar_colecao(n)))
    rodadas[n] = rag.rodar_referencia(n)
for n, r in rodadas.items():
    ok = sum(r["respostas_ok"].values())
    assert r["recall5_hibrido"] >= r["recall5_vetorial"], f"{n}: híbrido pior que vetorial"
    assert ok >= 20, f"{n}: só {ok}/25"
    assert r["respostas_ok"]["sem_resposta"] == 5 and r["respostas_ok"]["fora_do_escopo"] == 2, f"{n}: recusas erradas"
print("✅ seus testes passaram")

**Próximo:** Bancada 5 · Agentes. O RAG vira a ferramenta `consultar_politica` do agente, e a regra de permissão ganha uma versão mais séria: **nenhuma ação sem aprovação humana registrada**.